# Do mushroom photos contain a scale signal?

This notebook reads `../data/images/<mushroom_id>/` and joins photos to `../data/mushrooms.json`. It compares image-level properties (dimensions, color, brightness, texture, and file size) with the dataset's `properties.microscopic` field.

**Interpretation:** `microscopic` is a species identification property, not a label saying that a particular photo was taken through a microscope. Treat it as a proxy for exploration only. There is no image-level scale ground truth here, so a result can reveal candidate signals or dataset/source bias, but cannot establish a reliable micrograph detector.


In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    adjusted_rand_score, balanced_accuracy_score, ConfusionMatrixDisplay,
    classification_report, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("../data")
IMAGE_DIR = DATA_DIR / "images"


In [2]:
with (DATA_DIR / "mushrooms.json").open(encoding="utf-8") as f:
    dataset = json.load(f)

mushrooms = pd.json_normalize(dataset["mushrooms"], sep=".")
mushrooms = mushrooms.set_index("id", drop=False)
label_col = "properties.microscopic"

image_paths = [p for p in IMAGE_DIR.glob("*/*") if p.is_file()]
manifest = pd.DataFrame({
    "path": image_paths,
    "mushroom_id": [int(p.parent.name) if p.parent.name.isdigit() else None for p in image_paths],
    "extension": [p.suffix.lower() for p in image_paths],
    "bytes": [p.stat().st_size for p in image_paths],
    "mtime_ns": [p.stat().st_mtime_ns for p in image_paths],
})
manifest = manifest.dropna(subset=["mushroom_id"]).copy()
manifest["mushroom_id"] = manifest["mushroom_id"].astype(int)
manifest["species_microscopic_proxy"] = manifest["mushroom_id"].map(mushrooms[label_col])

print(f"Metadata records: {len(mushrooms):,}")
print(f"Image files: {len(manifest):,} across {manifest.mushroom_id.nunique():,} IDs")
print(f"Files with a metadata match and non-missing proxy label: {manifest.species_microscopic_proxy.notna().sum():,}")
print("Image extensions:")
display(manifest.extension.value_counts().to_frame("files"))
print("IDs with images by proxy label (counts are species IDs):")
display(manifest.drop_duplicates("mushroom_id").species_microscopic_proxy.value_counts(dropna=False).to_frame("IDs"))


Metadata records: 1,457
Image files: 44,148 across 1,457 IDs
Files with a metadata match and non-missing proxy label: 44,148
Image extensions:


,files
extension,
.jpg,43964
.jpeg,137
.png,47


IDs with images by proxy label (counts are species IDs):


,IDs
species_microscopic_proxy,
False,868
True,589


## Extract image-level descriptors

Images are EXIF-oriented and reduced to at most 256 pixels on the long edge before pixel statistics are computed. This keeps the pass manageable while retaining broad color and texture cues. Corrupt and unsupported files are counted and skipped. Results are cached in `image_features_cache.csv`; the cache is reused while the image paths, file sizes, modification times, and feature version are unchanged. Delete the cache files to force a fresh extraction.


In [ ]:
def describe_image(path):
    with Image.open(path) as original:
        image = ImageOps.exif_transpose(original).convert("RGB")
        width, height = image.size
        image.thumbnail((256, 256))
        rgb = np.asarray(image, dtype=np.float32) / 255.0

    gray = rgb.mean(axis=2)
    # Entropy and gradient features summarize coarse texture, not biological scale.
    hist, _ = np.histogram(gray, bins=32, range=(0, 1), density=False)
    probabilities = hist / max(hist.sum(), 1)
    probabilities = probabilities[probabilities > 0]
    entropy = -(probabilities * np.log2(probabilities)).sum()
    gy, gx = np.gradient(gray)
    gradient = np.hypot(gx, gy)
    saturation = rgb.max(axis=2) - rgb.min(axis=2)
    return {
        "width": width, "height": height,
        "log_pixels": np.log1p(width * height),
        "aspect_ratio": width / max(height, 1),
        "gray_entropy": entropy,
        "mean_brightness": float(gray.mean()),
        "brightness_std": float(gray.std()),
        "mean_saturation": float(saturation.mean()),
        "near_gray_fraction": float((saturation < 0.04).mean()),
        "mean_gradient": float(gradient.mean()),
        "gradient_p90": float(np.quantile(gradient, 0.90)),
    }

import hashlib
from concurrent.futures import ThreadPoolExecutor

CACHE_DIR = DATA_DIR.parent / ".cache"
FEATURE_CACHE = CACHE_DIR / "image_features_cache.csv"
CACHE_METADATA = CACHE_DIR / "image_features_cache.json"
FEATURE_VERSION = 1  # Bump this when describe_image changes.
MAX_WORKERS = 16
BATCH_SIZE = 1_000

def process_row(row):
    try:
        descriptors = describe_image(row.path)
        return ({
            "path": row.path, "mushroom_id": row.mushroom_id,
            "bytes": row.bytes, "mtime_ns": row.mtime_ns,
            "species_microscopic_proxy": row.species_microscopic_proxy,
            **descriptors,
        }, None)
    except (OSError, ValueError, UnidentifiedImageError) as error:
        return (None, (row.path, str(error)))

def current_manifest_signature(frame):
    digest = hashlib.sha256()
    for row in frame.sort_values("path").itertuples(index=False):
        digest.update(f"{row.path}\0{row.mushroom_id}\0{row.bytes}\0{row.mtime_ns}\0{row.species_microscopic_proxy!r}\n".encode())
    return digest.hexdigest()

manifest_signature = current_manifest_signature(manifest)
cache_info = {
    "feature_version": FEATURE_VERSION,
    "manifest_sha256": manifest_signature,
}
cache_is_current = False
if FEATURE_CACHE.exists() and CACHE_METADATA.exists():
    try:
        cache_is_current = json.loads(CACHE_METADATA.read_text(encoding="utf-8")) == cache_info
        if cache_is_current:
            image_features = pd.read_csv(FEATURE_CACHE)
            failures = json.loads(CACHE_METADATA.with_suffix(".failures.json").read_text(encoding="utf-8"))
            print(f"Loaded {len(image_features):,} cached image descriptors from {FEATURE_CACHE}")
    except (OSError, ValueError, json.JSONDecodeError):
        cache_is_current = False

if not cache_is_current:
    records, failures = [], []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        for start in range(0, len(manifest), BATCH_SIZE):
            batch = manifest.iloc[start : start + BATCH_SIZE]
            for record, failure in pool.map(process_row, batch.itertuples(index=False)):
                if record is not None:
                    records.append(record)
                if failure is not None:
                    failures.append((str(failure[0]), failure[1]))
            completed = min(start + len(batch), len(manifest))
            if completed % 5000 == 0 or completed == len(manifest):
                print(f"Read {completed:,}/{len(manifest):,} files")

    image_features = pd.DataFrame(records)
    # Write temporary files first so interrupted notebook runs do not leave a partial cache.
    feature_tmp = FEATURE_CACHE.with_suffix(".csv.tmp")
    metadata_tmp = CACHE_METADATA.with_suffix(".json.tmp")
    failures_path = CACHE_METADATA.with_suffix(".failures.json")
    failures_tmp = failures_path.with_suffix(".json.tmp")
    image_features.to_csv(feature_tmp, index=False)
    failures_tmp.write_text(json.dumps(failures), encoding="utf-8")
    metadata_tmp.write_text(json.dumps(cache_info, indent=2), encoding="utf-8")
    feature_tmp.replace(FEATURE_CACHE)
    failures_tmp.replace(failures_path)
    metadata_tmp.replace(CACHE_METADATA)
    print(f"Saved {len(image_features):,} image descriptors to {FEATURE_CACHE}")

print(f"Decoded: {len(image_features):,}; unreadable: {len(failures):,}")
if failures:
    display(pd.DataFrame(failures, columns=["path", "error"]).head(10))
display(image_features.head())

Read 5,000/44,148 files
Read 10,000/44,148 files
Read 15,000/44,148 files
Read 20,000/44,148 files


## Compare the proxy groups

The first comparison is at the image level; the ID-level summary below gives every mushroom equal weight, regardless of how many photos it has. Large differences can still reflect photographers, websites, compression, or species-specific photo collections.


In [ ]:
feature_cols = [
    "log_pixels", "aspect_ratio", "bytes", "gray_entropy",
    "mean_brightness", "brightness_std", "mean_saturation",
    "near_gray_fraction", "mean_gradient", "gradient_p90",
]

image_features["species_microscopic_proxy"] = image_features["species_microscopic_proxy"].astype("boolean")
summary = image_features.groupby("species_microscopic_proxy", dropna=True)[feature_cols].median().T
display(summary.rename(columns={False: "proxy=False", True: "proxy=True"}))

plot_features = ["log_pixels", "aspect_ratio", "bytes", "gray_entropy", "mean_saturation", "mean_gradient"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feature in zip(axes.flat, plot_features):
    groups = [image_features.loc[image_features.species_microscopic_proxy == value, feature].dropna()
              for value in (False, True)]
    ax.boxplot(groups, tick_labels=["False", "True"], showfliers=False)
    ax.set_title(feature)
    ax.set_xlabel("Species microscopic proxy")
fig.suptitle("Image descriptors by species metadata flag")
fig.tight_layout()
plt.show()

## Explore the manually annotated images with PCA

The annotations are image-level labels (`macroscopic` and `micro`). PCA is fit on the standardized descriptors for all successfully decoded images; the plot highlights the annotated subset. A two-cluster KMeans fit gives a simple unsupervised grouping to compare with those labels. PCA axes summarize variance and are not physical magnification measurements.


In [ ]:
annotation_path = DATA_DIR / "annotated.csv"
annotations = pd.read_csv(
    annotation_path,
    dtype={"id": "string", "image": "string", "label": "string"},
)
annotations["id"] = pd.to_numeric(annotations["id"], errors="coerce").astype("Int64").astype("string")
annotations["image"] = annotations["image"].str.strip()
annotations["label"] = annotations["label"].str.strip().str.lower().replace("", pd.NA)

print(f"Annotation rows: {len(annotations):,}")
display(annotations["label"].value_counts(dropna=False).rename_axis("label").to_frame("rows"))

# Join by both mushroom ID and basename; retain labels with no decoded image for an audit.
feature_rows = image_features.reset_index(names="feature_row").copy()
feature_rows["id"] = pd.to_numeric(feature_rows["mushroom_id"], errors="coerce").astype("Int64").astype("string")
feature_rows["image"] = feature_rows["path"].map(lambda path: Path(path).name)
labelled_annotations = annotations.dropna(subset=["id", "image", "label"]).copy()
if labelled_annotations.duplicated(["id", "image"]).any():
    raise ValueError("annotated.csv has duplicate labeled (id, image) rows; resolve them before plotting")

annotated_features = labelled_annotations.merge(
    feature_rows[["id", "image", "feature_row"]],
    on=["id", "image"], how="left", validate="one_to_one", indicator=True,
)
print(f"Labeled annotations: {len(labelled_annotations):,}; matched to extracted features: {(annotated_features['_merge'] == 'both').sum():,}")
print(f"Labeled annotations without readable local features: {(annotated_features['_merge'] != 'both').sum():,}")
annotated_features = annotated_features.dropna(subset=["feature_row"]).copy()
if annotated_features["label"].nunique() < 2:
    raise ValueError("PCA comparison needs at least two labeled classes")

# Log file size to reduce its right skew, then standardize each feature before PCA/KMeans.
X_all = image_features[feature_cols].astype(float).replace([np.inf, -np.inf], np.nan)
X_all["bytes"] = np.log1p(X_all["bytes"])
valid_rows = X_all.notna().all(axis=1)
X_all = X_all.loc[valid_rows]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)
pca = PCA(n_components=2, random_state=42)
coordinates = pca.fit_transform(X_scaled)
clusters = KMeans(
    n_clusters=annotated_features["label"].nunique(), random_state=42, n_init=10
).fit_predict(X_scaled)
projection = pd.DataFrame({
    "feature_row": X_all.index, "PC1": coordinates[:, 0], "PC2": coordinates[:, 1],
    "cluster": clusters,
})
annotated_projection = annotated_features.merge(
    projection, on="feature_row", how="inner", validate="one_to_one"
)
print(f"Explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}; total={pca.explained_variance_ratio_.sum():.1%}")
print(f"KMeans vs. manual labels (adjusted Rand index): {adjusted_rand_score(annotated_projection['label'], annotated_projection['cluster']):.3f}")

label_colors = {"macroscopic": "#287a55", "micro": "#cf5c36"}
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(coordinates[:, 0], coordinates[:, 1], s=5, c="#d8d8d8", alpha=0.18, label="unlabeled / background")
for label, group in annotated_projection.groupby("label"):
    axes[0].scatter(group["PC1"], group["PC2"], s=24, alpha=0.8,
                    color=label_colors.get(label), label=label)
axes[0].set_title("Manual annotation")
axes[0].legend(markerscale=1.5)

for cluster_id in sorted(projection["cluster"].unique()):
    group = projection[projection["cluster"] == cluster_id]
    axes[1].scatter(group["PC1"], group["PC2"], s=5, alpha=0.18, label=f"cluster {cluster_id}")
for label, group in annotated_projection.groupby("label"):
    axes[1].scatter(group["PC1"], group["PC2"], s=24, facecolors="none",
                    edgecolors=label_colors.get(label), linewidths=0.8, label=f"labeled: {label}")
axes[1].set_title("Unsupervised groups with manual labels overlaid")
axes[1].legend(markerscale=1.5, fontsize=8)
for ax in axes:
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
fig.tight_layout()
plt.show()

loadings = pd.DataFrame(pca.components_.T, index=feature_cols, columns=["PC1", "PC2"])
display(loadings.assign(max_abs_loading=loadings.abs().max(axis=1))
        .sort_values("max_abs_loading", ascending=False).drop(columns="max_abs_loading"))


## Can the image descriptors predict the species flag?

This is a diagnostic for whether the image collection carries a repeatable signal associated with the metadata flag. Each row below is one mushroom ID, with image descriptors summarized by their median. The split is by ID, so photos from the same ID cannot appear on both sides. Performance does **not** measure micrograph detection because the target is not an image-level scale annotation.


In [ ]:
id_features = (
    image_features.dropna(subset=["species_microscopic_proxy"])
    .groupby(["mushroom_id", "species_microscopic_proxy"], as_index=False)[feature_cols]
    .median()
)
X = id_features[feature_cols].replace([np.inf, -np.inf], np.nan)
y = id_features.species_microscopic_proxy.astype(bool)
valid = X.notna().all(axis=1)
X, y, ids = X.loc[valid], y.loc[valid], id_features.loc[valid, "mushroom_id"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model = make_pipeline(
    StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000)
)
model.fit(X_train, y_train)
prediction = model.predict(X_test)
probability = model.predict_proba(X_test)[:, 1]
print(f"Held-out IDs: {len(y_test)} (train {len(y_train)}); ID overlap: {len(set(ids.loc[X_train.index]) & set(ids.loc[X_test.index]))}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_test, prediction):.3f}")
print(f"ROC AUC: {roc_auc_score(y_test, probability):.3f}")
print(classification_report(y_test, prediction, target_names=["proxy false", "proxy true"]))
ConfusionMatrixDisplay.from_predictions(y_test, prediction, display_labels=["proxy false", "proxy true"]);
plt.title("Held-out mushroom IDs")
plt.show()

coefs = pd.Series(model[-1].coef_[0], index=feature_cols).sort_values()
coefs.plot.barh(figsize=(8, 5), title="Standardized logistic coefficients (proxy flag)")
plt.xlabel("Coefficient")
plt.show()


## Inspect actual image examples

Reviewing examples is essential: numeric descriptors can be driven by unrelated collection artifacts. The contact sheet samples IDs from each proxy group and shows one image per ID. If these labels do not correspond to visible microscopy photos, annotate image scale explicitly before fitting a detector.


In [ ]:
rng = np.random.default_rng(42)
ids_by_proxy = (image_features.dropna(subset=["species_microscopic_proxy"])
                .groupby(["mushroom_id", "species_microscopic_proxy"])["path"]
                .first().reset_index())
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for row_i, label in enumerate((False, True)):
    candidates = ids_by_proxy[ids_by_proxy.species_microscopic_proxy == label]
    picks = candidates.sample(min(8, len(candidates)), random_state=42)
    for ax, item in zip(axes[row_i], picks.itertuples(index=False)):
        with Image.open(item.path) as im:
            ax.imshow(ImageOps.exif_transpose(im).convert("RGB"))
        ax.set_title(f"ID {item.mushroom_id}")
        ax.axis("off")
    for ax in axes[row_i, len(picks):]:
        ax.axis("off")
    axes[row_i, 0].set_ylabel(f"proxy={label}", rotation=0, labelpad=40, va="center")
fig.suptitle("One random-by-ID photo per metadata group")
fig.tight_layout()
plt.show()


## What to conclude

- Compare the ID-level held-out scores with the 0.50 balanced-accuracy baseline and check the contact sheet before treating any association as useful.
- Strong prediction of `properties.microscopic` may be a source or species signal; it does not prove that photos show microscope views.
- To train and evaluate a normal-scale versus microscope-image filter, add an image-level annotation (for example, `normal`, `microscope`, `uncertain`) for a representative sample, then split evaluation by mushroom ID and preferably by image source.
